# 07 · Build a geometry–dynamics atlas

**ScGeo 0.2.0 learning track.** An integrated atlas combines existing evidence without rerunning an analysis inside the renderer. Minimal AnnData with an embedding is enough for the state-only view.

Use the [separate tutorial environment](README.md). Run cells in order in a fresh kernel. Outputs are written to a new ignored results directory; manuscript inputs and outputs are not used.

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import scgeo as sg
from IPython.display import display

cwd = Path.cwd().resolve()
root = next((p for p in (cwd, *cwd.parents)
             if (p / "scripts/scgeo_v020_examples/SOURCE_PART1.json").is_file()), None)
if root is None:
    raise RuntimeError("Open this notebook from the scgeo-notebooks checkout.")
examples = root / "scripts/scgeo_v020_examples"
if str(examples) not in sys.path:
    sys.path.insert(0, str(examples))  # only tutorial helpers, not package source
from tutorial_support import setup, save_preview, show_gallery, FORMATS
out = setup(root, "07_geometry_dynamics_atlas")

In [ ]:
from anndata import AnnData
from scipy import sparse
from plotting_gallery import make_gallery_data
from geometry_dynamics_atlas_gallery import run_gallery
adata = make_gallery_data(seed=17)
states = ["State A", "State B", "State C"]
adata.obs["state"] = pd.Categorical(adata.obs["state"], categories=states)
adata.uns["paga"] = {"groups":"state", "connectivities":sparse.csr_matrix(
    [[0,.62,.29],[.62,0,.72],[.29,.72,0]])}
sg.tl.robust_shift(adata, rep="X_toy", by="state", condition_key="condition",
    group0="control", group1="treated", sample_key="sample",
    center="geometric_median", n_boot=35, seed=17)
report = sg.get.state_report(adata, node_key="state")
result = sg.pl.geometry_dynamics_atlas(
    adata, basis="display", color="state", node_key="state",
    condition_key="condition", group0="control", group1="treated",
    sample_key="sample", report=report,
    panels=("states","velocity","paga","evidence"), velocity_mode="stream",
    velocity_options={"grid_size":26, "n_neighbors":28, "min_neighbors":6, "min_coherence":.12},
    paga_options={"graph_node_order":states, "min_cells":10},
    highlight_states=["State A"], return_result=True, show=False)
save_preview(result, out, "full_atlas")

## Minimal object: no dynamics and no graph

Do not fabricate missing panels. The auto layout recognizes the actual supplied inputs.

In [ ]:
simple = AnnData(obs=adata.obs[["state"]].copy())
simple.obsm["X_display"] = adata.obsm["X_display"].copy()
result = sg.pl.geometry_dynamics_atlas(simple, basis="display", color="state",
                                      panels="auto", show=False)
assert result.metadata["panels"] == ["states"]
save_preview(result, out, "minimal_atlas")

The [complete atlas example](../../../scripts/scgeo_v020_examples/geometry_dynamics_atlas_gallery.py) also accepts checksum-matched public inputs. All graph edges, vectors and coordinates above are simulated. The [atlas guide](https://github.com/liuifrec/scgeo/blob/v0.2.0/docs/GEOMETRY_DYNAMICS_ATLAS.md) describes panel requirements.